# Hard-sector posterior and reference parameters

Read the [included six-parameter sample archive](MCMCSamples.txt.bz2), plot posterior marginals, convert the supplied reference point to JETSCAPE XML parameters, and estimate a joint KDE mode. The file contains natural logarithms of the three coefficients. The samples were provided by lead authors of the [related study](https://arxiv.org/abs/2408.08247); please cite that paper when using them. Dataset fields and processing details are described in [metadata.json](metadata.json). The supplied reference is a sample-based KDE estimate, not a MAP vector quoted from a published table.

In [ ]:
from pathlib import Path
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/inference_tools.py").is_file())
sys.path.insert(0, str(ROOT / "scripts"))
from inference_tools import load_samples, summarize, corner_plot, approximate_joint_mode, hard_point_to_xml, sha256
OUT = ROOT / "outputs" / "inference"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
DATA = ROOT / "hard"
metadata = json.loads((DATA / "metadata.json").read_text())
names = metadata["columns"]
path = DATA / "MCMCSamples.txt.bz2"
assert sha256(path) == metadata["archive_sha256"]
samples = load_samples(path, names)
assert list(samples.shape) == metadata["shape"]
print("Samples, coordinates:", samples.shape)
summary = summarize(samples, names)
print(json.dumps(summary, indent=2))
(OUT / "hard_summary.json").write_text(json.dumps(summary, indent=2) + "\n")

The red lines mark the supplied joint KDE reference point in [reference_point.json](reference_point.json). Plotting uses a seeded subset of up to 20,000 rows; summaries use every row. Repeated MCMC samples are retained, and no prior cuts, additional burn-in removal or thinning are applied. These plots alone do not establish convergence.

In [ ]:
reference = json.loads((DATA / "reference_point.json").read_text())
labels = [r"$\alpha_s$", r"$Q_0$ [GeV]", r"$\ln c_1$", r"$\ln c_2$", r"$\tau_0$ [fm/$c$]", r"$\ln c_3$"]
fig = corner_plot(samples, names, labels=labels, reference=reference["point"], title="Hard-sector posterior samples (log coefficients)")
fig.savefig(OUT / "hard_posterior.png", dpi=130)
plt.show()

## Convert the reference point into simulation coefficients

For a point represented by $\ln c_i$, the XML coefficients are $c_i=\exp(\ln c_i)$. The conversion function uses explicitly declared coordinates; apply the exponential once to obtain physical coefficients. These are example reference settings; use the appropriate calibration and collision-system settings for a physics calculation.

In [ ]:
reference_xml = hard_point_to_xml(reference["point"], input_coordinates=names)
print(json.dumps(reference_xml, indent=2))
assert np.allclose([reference_xml[k] for k in ["qhatA", "qhatB", "qhatC"]], [.010992742463249567, .26936713924688244, 39.3824387524629])
(OUT / "hard_reference_xml_parameters.json").write_text(json.dumps(reference_xml, indent=2) + "\n")

## Estimate a joint mode from samples

This bounded calculation fits a Gaussian KDE to 4,000 rows and evaluates it at 1,500 candidate samples, using seed 123. It illustrates a sample-based density estimate and need not reproduce the supplied reference. It cannot evaluate or optimize the original posterior. Assess bandwidth, boundary bias and candidate coverage before using a KDE mode as a reference estimate.

In [ ]:
estimate = approximate_joint_mode(samples, names, fit_size=4000, candidates=1500, seed=123)
print(json.dumps(estimate, indent=2))
(OUT / "hard_joint_kde_estimate.json").write_text(json.dumps(estimate, indent=2) + "\n")

A marginal median, a marginal mode, a joint KDE mode and the highest-posterior retained sample are different quantities. `highest_posterior_sample` is available when aligned posterior scores are supplied. Converting a point into $c_i$ does not find the mode of the density in those coordinates: $p(c_1,c_2,c_3,\ldots)=p(\ln c_1,\ln c_2,\ln c_3,\ldots)/(c_1c_2c_3)$. Preserve the coordinate declaration and estimation method with each result.